# Detecting Motor Anomalies

Preventing motor anomalies is a bit more complicated than battery issues. Usually, motors operate in a certain range of power, but sometimes they may present anomalous behavior. Their power consumption can go to high, due to environmental issues, or too low, due to aging issues.

As usual, let's start by recovering and looking at data:

In [ ]:
%store -r data
data.head()

In [ ]:
%store -r bucket
bucket

# Exploratory Data Analysis

In [ ]:
train_data = data[["motor_peak_mA"]]
train_data = train_data[train_data["motor_peak_mA"] > 0]
train_data.head()

In [ ]:
train_data.describe()

In [ ]:
import matplotlib.pyplot as plt
train_data.plot(rot=30)

## Synthetic Ground Truth

In [ ]:
anomalies = data[["motor_peak_mA"]]
anomalies = anomalies[anomalies["motor_peak_mA"] > 0]
anomalies.info()

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_dataframe = train_test_split(anomalies, test_size=0.2)

In [ ]:
test_data = test_dataframe.copy()
test_data["anomaly"] = test_data["motor_peak_mA"] > 4000
test_data["anomaly"] = test_data["anomaly"] | (test_data["motor_peak_mA"] > 50) & (test_data["motor_peak_mA"] < 200)
test_data["anomaly"] = test_data["anomaly"].astype(int)
test_data.groupby("anomaly").count().head()

In [ ]:
test_data.describe()

In [ ]:
train_data.describe()

# Random Cut Forest Training

In [ ]:
train_array = train_data.values
train_array

In [ ]:
test_array = test_data[["motor_peak_mA"]].values
test_array

In [ ]:
labels_array = test_data["anomaly"].values
labels_array

In [ ]:
import io
import boto3

s3bucket = boto3.resource('s3').Bucket(bucket)

def upload_csv(array, key):
    buf = io.BytesIO()
    for row in array.astype('float32'):
        buf.write((','.join(str(v) for v in row) + '\n').encode())
    buf.seek(0)
    s3bucket.Object(key).upload_fileobj(buf)

In [ ]:
from sagemaker.core.training.configs import InputData

prefix = "mt-motor-anomaly"

train_key = "{}/input/{}".format(prefix, "train.csv")
test_key  = "{}/input/{}".format(prefix, "test.csv")

upload_csv(train_array, train_key)
upload_csv(test_array, test_key)

train_input = InputData(
    channel_name="train",
    data_source="s3://{}/{}".format(bucket, train_key),
)

test_input = InputData(
    channel_name="test",
    data_source="s3://{}/{}".format(bucket, test_key),
)

rcf_input = [train_input, test_input]
rcf_input

# RCF Training

In [ ]:
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core import image_uris

sagemaker_session = Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()
rcf_container = image_uris.retrieve(framework='randomcutforest', region=region)
rcf_container

In [ ]:
rcf_hparams = {
    "num_samples_per_tree": 512,
    "num_trees": 50,
    "feature_dim": 1,
    "eval_metrics": "accuracy"
}

In [ ]:
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.core.training.configs import Compute, OutputDataConfig

rcf_trainer = ModelTrainer(
    training_image=rcf_container,
    role=role,
    compute=Compute(
        instance_count=1,
        instance_type="ml.m5.large",
    ),
    output_data_config=OutputDataConfig(
        s3_output_path="s3://{}/{}/output".format(bucket, prefix),
    ),
    base_job_name="mt-motor-anomaly",
    hyperparameters=rcf_hparams,
)

In [ ]:
rcf_trainer.train(input_data_config=rcf_input)

In [ ]:
print('Training job name: {}'.format(rcf_trainer.latest_training_job.training_job_name))

In [ ]:
from sagemaker.serve.model_builder import ModelBuilder

model_builder = ModelBuilder(
    model=rcf_trainer,
    role_arn=role,
)
model_builder.build()
rcf_endpoint = model_builder.deploy(
    instance_type="ml.m5.large",
    initial_instance_count=1,
)

In [ ]:
rcf_inference_endpoint = rcf_endpoint.endpoint_name
%store rcf_inference_endpoint
rcf_inference_endpoint

In [ ]:
sample_data = train_data[:5].values
sample_data

In [ ]:
import json

csv_body = "\n".join(",".join(str(v) for v in row) for row in sample_data)
response = rcf_endpoint.invoke(body=csv_body, content_type="text/csv", accept="application/json")
results = json.loads(response.body.read().decode('utf-8'))
results

In [ ]:
import pandas as pd
sigmas = 1

scores = results["scores"]
scores = [score["score"] for score in scores]
series = pd.Series(scores)
score_mean = series.mean()
score_max = series.max()
score_std = series.std()
score_cutoff = score_mean + sigmas * score_std
(score_mean, score_max, score_std, score_cutoff)

In [ ]:
anomalies = series[series > score_cutoff]
anomalies

In [ ]:
"{} anomalies detected".format(len(anomalies))

## Motor Maintenance

Now that we can detect anomalies in past data, let's combine that with forecasting for predictive [motor maintenance](mt-motor-maintenance.ipynb).